1. 📝 Why Do We Need a Transformer Block?

Self-attention lets tokens exchange information:

Token representations
        ↓
Self-Attention
        ↓
Context-aware representations

But attention alone isn't enough.

A Transformer block adds:

Multi-Head Self-Attention
Residual connection
Layer Normalization
Feed-Forward Network
Another residual connection
Another Layer Normalization

Conceptually:

             Input X
                │
                ▼
       Multi-Head Attention
                │
                ▼
          Add X (Residual)
                │
                ▼
          LayerNorm
                │
                ▼
        Feed-Forward Network
                │
                ▼
          Add Residual
                │
                ▼
          LayerNorm
                │
                ▼
              Output

📝 Residual Connections

A residual connection simply adds the original input back:

$$ Y = X + F(X) $$

Why?

Because during deep-network training, information and gradients can become difficult to propagate.

The shortcut gives the network a direct path.

In [1]:
import torch

X=torch.tensor([
    [1.0,2.0],
    [3.0,4.0]
])

F_X=torch.tensor([
    [0.5,1.0],
    [1.5,2.0]
])

Y=X+F_X
print(Y)

tensor([[1.5000, 3.0000],
        [4.5000, 6.0000]])


Layer Normalization

After transformations, the values can have different distributions.

LayerNorm normalizes features for each token.

Conceptually:

$$ \hat{x} = \frac{x-\mu}{\sqrt{\sigma^2+\epsilon}} $$

Then learnable parameters scale and shift the result:

y=γ
x
^
+β

In [2]:
import torch.nn as nn

layer_norm=nn.LayerNorm(4)

x=torch.tensor([
    [1.0,2.0,3.0,4.0],
    [10.0,20.0,30.0,40.0]
])

y=layer_norm(x)

print(y)

tensor([[-1.3416, -0.4472,  0.4472,  1.3416],
        [-1.3416, -0.4472,  0.4472,  1.3416]],
       grad_fn=<NativeLayerNormBackward0>)


eed-Forward Network

After attention, each token independently passes through a small neural network.

Usually:

$$ FFN(x)=W_2\,\sigma(W_1x+b_1)+b_2 $$

Modern Transformers commonly use an activation such as GELU.

The structure is:

Input
  ↓
Linear
  ↓
GELU
  ↓
Linear
  ↓
Output

Typically the hidden dimension is larger than the model dimension.

Example:

512→2048→512

In [4]:
ffn=nn.Sequential(
    nn.Linear(4,16),
    nn.GELU(),
    nn.Linear(16,4)
)

x=torch.randn(2,3,4)

y=ffn(x)

print("Input: ",x.shape)
print("Output: ",y.shape)

Input:  torch.Size([2, 3, 4])
Output:  torch.Size([2, 3, 4])


Dropout

Dropout randomly disables some activations during training.

Training:
some values → 0

Inference:
all values used

It helps reduce overfitting.

In [5]:
dropout=nn.Dropout(0.1)

x=torch.ones(10)

print(dropout(x))

tensor([0.0000, 1.1111, 1.1111, 1.1111, 1.1111, 1.1111, 1.1111, 1.1111, 1.1111,
        1.1111])


Complete Transformer Block

Now combine everything:

              X
              │
              ▼
       Multi-Head Attention
              │
              ▼
           + X
              │
              ▼
          LayerNorm
              │
              ▼
             FFN
              │
              ▼
           + residual
              │
              ▼
          LayerNorm
              │
              ▼
           Output

In [6]:
class TransformerBlock(nn.Module):

  def __init__(self, d_model, num_heads, d_ff):
    super().__init__()

    self.attention=nn.MultiheadAttention(
        d_model,
        num_heads,
        batch_first=True
    )

    self.norm1=nn.LayerNorm(d_model)
    self.norm2=nn.LayerNorm(d_model)

    self.ffn=nn.Sequential(
        nn.Linear(d_model,d_ff),
        nn.GELU(),
        nn.Linear(d_ff,d_model)
    )

  def forward(self,x,casual_mask=None):
    attn_output, _=self.attention(
        x,x,x,
        attn_mask=casual_mask
    )

    x=self.norm1(x+attn_output)
    ffn_output=self.ffn(x)
    x-self.norm2(x+ffn_output)

    return x

In [7]:
block = TransformerBlock(
    d_model=8,
    num_heads=2,
    d_ff=32
)

x = torch.randn(2, 5, 8)

output = block(x)

print("Input :", x.shape)
print("Output:", output.shape)

Input : torch.Size([2, 5, 8])
Output: torch.Size([2, 5, 8])
